# kNN classifier – Breast Cancer Wisconsin (Diagnostic)

Tässä notebookissa rakennetaan k-nearest neighbors (kNN) -luokittelija UCI:n **Breast Cancer Wisconsin (Diagnostic)** -aineistolle. Työ on jäsennelty CRISP-DM-mallin kuuteen vaiheeseen:

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation
6. Deployment

Tavoitteena ei ole vain saada mahdollisimman korkea accuracy, vaan myös tulkita virheiden merkitystä. Erityisesti pahanlaatuisen kasvaimen luokittelu hyvänlaatuiseksi (false negative) on tämän ongelman kannalta vakavampi virhe kuin false positive.

# 1. Business Understanding

Tehtävänä on rakentaa binäärinen kNN-luokittelija, joka ennustaa rintakasvaimen diagnoosin:

- **M = malignant (pahanlaatuinen)**
- **B = benign (hyvänlaatuinen)**

Aineiston piirteet on laskettu rintamassasta otetun **fine needle aspirate (FNA)** -näytteen digitoidusta kuvasta ja ne kuvaavat kuvassa näkyvien soluytimien ominaisuuksia.

### Onnistumisen arviointi

Mallia arvioidaan useasta näkökulmasta:

- **Accuracy:** kuinka suuri osa kaikista havainnoista luokitellaan oikein.
- **Precision (M):** kuinka suuri osa malignant-luokkaan ennustetuista on todella malignant.
- **Recall (M):** kuinka suuri osa todellisista malignant-tapauksista tunnistetaan.
- **Confusion matrix:** näyttää oikeiden ja väärien luokitusten määrät erikseen.

Pelkkä accuracy ei ole tässä riittävä mittari. Erityisesti **malignant → benign** -virhe eli false negative on tärkeä havaita, koska tällöin pahanlaatuinen tapaus jää mallilta tunnistamatta.

Tämä notebook on koneoppimisen harjoitustyö, ei kliiniseen käyttöön validoitu diagnostiikkajärjestelmä.

# 2. Data Understanding

Aineisto haetaan UCI Machine Learning Repositorysta (`id=17`). Aineistossa on:

- **569 havaintoa**
- **30 jatkuvaa numeerista input-muuttujaa**
- **Diagnosis** target-muuttujana
- lisäksi **ID**, joka on tunniste eikä mallin selittävä muuttuja
- UCI:n kuvauksen mukaan aineistossa **ei ole puuttuvia arvoja**

Kymmenen soluytimen perusominaisuutta esiintyy kolmena mittausryhmänä (`1`, `2`, `3`), jolloin input-piirteitä on yhteensä 30.

| Feature family | Merkitys |
|---|---|
| radius | Etäisyyksien keskiarvo keskipisteestä reunaviivan pisteisiin |
| texture | Harmaasävyarvojen keskihajonta |
| perimeter | Soluytimen ympärysmitta |
| area | Soluytimen pinta-ala |
| smoothness | Paikallinen vaihtelu säteen pituuksissa |
| compactness | Tiiviys, määritelty perimeter² / area − 1 |
| concavity | Reunaviivan koverien osien voimakkuus |
| concave points | Koverien osien/pisteiden määrä |
| symmetry | Soluytimen symmetria |
| fractal dimension | “Coastline approximation” -tyyppinen mitta |

Ennen mallintamista tarkastetaan aineiston rakenne, muuttujatyypit, puuttuvat arvot, diagnoosiluokkien jakauma sekä numeeristen muuttujien tunnuslukuja.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

# Fetch dataset
breast_cancer = fetch_ucirepo(id=17)

# Input variables and target
X = breast_cancer.data.features.copy()
y = breast_cancer.data.targets["Diagnosis"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# UCI metadata and variable information
print(breast_cancer.metadata)
display(breast_cancer.variables)

In [ ]:
# Combine features and target for exploratory inspection
df = pd.concat([X, y], axis=1)

display(df.head())
print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

print("\nDiagnosis distribution:")
display(y.value_counts())
display(y.value_counts(normalize=True).rename("proportion"))

print("\nDescriptive statistics:")
display(X.describe().T)

### Data Understanding – tulkinta

Aineiston piirteet ovat numeerisia, mutta niiden mittakaavat poikkeavat selvästi toisistaan: esimerkiksi pinta-alan ja ympärysmitan arvot ovat aivan eri suuruusluokassa kuin smoothness- tai fractal dimension -muuttujat. Tämä on erityisen tärkeää kNN:lle, koska algoritmi perustuu havaintojen välisiin etäisyyksiin.

Myös diagnoosiluokkien jakauma kannattaa tarkistaa. Jos luokat eivät ole yhtä suuria, accuracy voi yksin antaa liian optimistisen kuvan mallista. Siksi arvioinnissa tarkastellaan erikseen malignant-luokan precisionia ja recallia.

# 3. Data Preparation

kNN perustuu havaintojen väliseen etäisyyteen. Jos muuttujat jätetään alkuperäisiin mittakaavoihinsa, suurempia lukuarvoja sisältävät muuttujat voivat hallita etäisyyslaskentaa. Tämän vuoksi input-muuttujat **standardoidaan StandardScalerilla**.

Data jaetaan ensin hold-out-periaatteella:

- **80 % training data**
- **20 % test data**
- `stratify=y` säilyttää diagnoosiluokkien suhteet mahdollisimman samanlaisina molemmissa osissa.
- `random_state=42` tekee jaosta toistettavan.

Tärkeää on sovittaa (`fit`) StandardScaler **vain training-dataan**. Testidata muunnetaan training-datasta opituilla keskiarvoilla ja keskihajonnoilla. Näin testidatan tietoa ei vuoda esikäsittelyvaiheessa mallin koulutukseen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining class distribution:")
display(y_train.value_counts(normalize=True))

print("Test class distribution:")
display(y_test.value_counts(normalize=True))

In [ ]:
scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using training-data parameters
X_test_scaled = scaler.transform(X_test)

# Optional DataFrames for easier inspection
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

display(X_train_scaled_df.head())

### Standardoinnin tarkistus

Training-datan standardoitujen muuttujien keskiarvojen pitäisi olla hyvin lähellä nollaa ja keskihajontojen lähellä yhtä. Testidatan ei tarvitse saada täsmälleen samoja arvoja, koska scaleria ei soviteta siihen erikseen.

In [ ]:
scaling_check = pd.DataFrame({
    "train_mean": X_train_scaled_df.mean(),
    "train_std": X_train_scaled_df.std(ddof=0)
})

display(scaling_check.head(10))

# 4. Modeling

kNN luokittelee uuden havainnon sen lähimpien naapureiden perusteella. Hyperparametri **k** määrää, kuinka monta lähintä naapuria päätöksessä käytetään.

- Pieni k reagoi voimakkaasti yksittäisiin havaintoihin ja voi olla herkkä kohinalle.
- Suurempi k tasoittaa päätösrajaa, mutta liian suuri k voi hävittää paikallista rakennetta.
- Binäärisessä luokittelussa kokeillaan tässä parittomia k-arvoja, mikä vähentää tasatilanteiden mahdollisuutta.

Tehtävänannon mukaisesti kokeillaan useita k:n arvoja ja raportoidaan niiden testitarkkuus.

In [ ]:
k_values = list(range(1, 20, 2))
results = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    y_pred_k = model.predict(X_test_scaled)

    results.append({
        "k": k,
        "accuracy": accuracy_score(y_test, y_pred_k),
        "precision_M": precision_score(y_test, y_pred_k, pos_label="M"),
        "recall_M": recall_score(y_test, y_pred_k, pos_label="M")
    })

results_df = pd.DataFrame(results)
display(results_df.style.format({
    "accuracy": "{:.3f}",
    "precision_M": "{:.3f}",
    "recall_M": "{:.3f}"
}))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results_df["k"], results_df["accuracy"], marker="o")
plt.xlabel("k (number of neighbours)")
plt.ylabel("Accuracy")
plt.title("kNN accuracy with different k values")
plt.xticks(k_values)
plt.grid(alpha=0.3)
plt.show()

### k-arvon valinnan tulkinta

Tulosta ei kannata arvioida vain yhden luvun perusteella. Accuracy kertoo kokonaisonnistumisesta, mutta tämän ongelman kannalta malignant-luokan recall on erityisen kiinnostava, koska se kertoo kuinka moni pahanlaatuinen tapaus löydetään.

Alla valitaan kokeilluista vaihtoehdoista korkean accuracyn antava k. Jos useampi k saa saman accuracyn, valitaan niistä se, jolla malignant-recall on paras ja tämän jälkeen pienempi k.

**Menetelmällinen huomio:** tässä harjoituksessa k-arvoja vertaillaan hold-out-testijoukolla tehtävänannon mukaisesti. Tiukemmassa koneoppimisprosessissa hyperparametri k valittaisiin erillisellä validation-datalla tai cross-validationilla, ja testijoukko käytettäisiin vasta lopulliseen, yhteen arviointiin. Muuten testijoukkoon voidaan epäsuorasti ylisovittaa k:n valinnan kautta.

In [ ]:
best_row = (
    results_df
    .sort_values(
        ["accuracy", "recall_M", "k"],
        ascending=[False, False, True]
    )
    .iloc[0]
)

best_k = int(best_row["k"])
print("Selected k:", best_k)
print(best_row)

# 5. Evaluation

Valittu malli koulutetaan training-datalla ja arvioidaan hold-out-testidatalla. Raportoidaan tehtävänannon vaatimat:

- confusion matrix
- accuracy
- precision
- recall

Lisäksi näytetään F1-score ja koko classification report, jotta molempien luokkien käyttäytyminen näkyy.

In [ ]:
final_model = KNeighborsClassifier(n_neighbors=best_k)
final_model.fit(X_train_scaled, y_train)
y_pred = final_model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred, labels=["B", "M"])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["B (benign)", "M (malignant)"]
)
disp.plot()
plt.title(f"Confusion matrix (k={best_k})")
plt.show()

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision_m = precision_score(y_test, y_pred, pos_label="M")
recall_m = recall_score(y_test, y_pred, pos_label="M")
f1_m = f1_score(y_test, y_pred, pos_label="M")

print(f"Accuracy:    {accuracy:.3f}")
print(f"Precision M: {precision_m:.3f}")
print(f"Recall M:    {recall_m:.3f}")
print(f"F1-score M:  {f1_m:.3f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix breakdown with B as negative class and M as positive class
tn, fp, fn, tp = cm.ravel()

print("True negatives  (B -> B):", tn)
print("False positives (B -> M):", fp)
print("False negatives (M -> B):", fn)
print("True positives  (M -> M):", tp)

### Tulosten tulkinta

Confusion matrixia tulkitaan tässä siten, että **malignant (M) on positiivinen luokka**:

- **TP:** malignant ennustettiin malignantiksi.
- **TN:** benign ennustettiin benigniksi.
- **FP:** benign ennustettiin malignantiksi.
- **FN:** malignant ennustettiin benigniksi.

Erityisesti FN-luku on tärkeä, koska se kuvaa pahanlaatuisia tapauksia, jotka malli jätti tunnistamatta.

Mittareiden merkitys:

\[
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
\]

\[
Precision = \frac{TP}{TP + FP}
\]

\[
Recall = \frac{TP}{TP + FN}
\]

Korkea accuracy kertoo hyvästä kokonaisluokittelusta, mutta kliinisen kaltaisessa ongelmassa korkea malignant-recall on erityisen tärkeä. Precision puolestaan kertoo, kuinka luotettava malignant-ennuste on. Mallin tuloksia pitää siksi tarkastella näiden mittareiden yhdistelmänä eikä vain accuracyn perusteella.

Hold-out-tulos riippuu myös siitä, mitkä havainnot päätyvät training- ja testijoukkoihin. `random_state` tekee juuri tämän analyysin toistettavaksi, mutta yksi jako ei yksin osoita mallin yleistä suorituskykyä.

# 6. Deployment

Mallia voitaisiin teknisesti käyttää uusien havaintojen luokitteluun, mutta käyttö vaatisi vähintään seuraavan prosessin:

1. Uudesta FNA-näytteestä täytyy muodostaa täsmälleen samat 30 input-piirrettä.
2. Piirteiden järjestyksen ja merkityksen täytyy vastata training-dataa.
3. Uudet arvot täytyy standardoida **samalla training-datasta sovitetulla scalerilla**.
4. Standardoidut arvot annetaan koulutetulle kNN-mallille.
5. Mallin ennuste ja sen epävarmuus/rajoitteet täytyy huomioida käyttökontekstissa.

### Rajoitteet

- Aineisto on melko pieni (569 havaintoa).
- Yksi hold-out-jako ei riitä osoittamaan mallin yleistyvyyttä.
- kNN:n ennustaminen voi muuttua raskaaksi, jos training-aineisto kasvaa paljon, koska uusia havaintoja verrataan tallennettuihin training-havaintoihin.
- Hyperparametrin valinta olisi luotettavampaa cross-validationilla.
- Todellista kliinistä käyttöä varten tarvittaisiin huomattavasti laajempi ja riippumaton validointi.

### Johtopäätös

kNN soveltuu tähän harjoitukseen hyvin, koska aineisto on suhteellisen pieni ja ongelma on selkeä binäärinen luokittelutehtävä. Standardointi on olennainen osa prosessia, koska kNN perustuu etäisyyksiin. Mallin onnistumista ei tule arvioida vain accuracyn avulla: erityisesti malignant-luokan recall ja false negative -tapaukset ovat keskeisiä.

Jatkokehityksessä tuloksia voisi vahvistaa esimerkiksi cross-validationilla, erillisellä hyperparametrien optimoinnilla ja vertaamalla kNN:ää muihin luokittelualgoritmeihin.